# Chapter 8. Volunteered Geographic Information and OpenStreetMap

*Starting With What You Have: A Quantitative Field Guide for Urban Research in Data-Scarce Settings*

Runs in a browser with no installation. Open in Google Colab and choose Runtime, then Run all.


## Step 0. Installation

To pull real OSM data, `osmnx` is the reproducible route and overpass-turbo.eu the browser route. Request metadata, since quality assessment needs contributor counts and edit dates.

In [ ]:
!pip install -q geopandas matplotlib

## Step 1. Load the data

Convert to a metre-based CRS first. Areas, lengths, and distances are all computed here and every one is meaningless in latitude and longitude.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import vgi_toolkit as v

bld = gpd.read_file("data/osm_buildings.geojson")
roads = gpd.read_file("data/osm_roads.geojson")
zones = gpd.read_file("data/zones.geojson")
print("buildings:", len(bld), "| roads:", len(roads), "| zones:", len(zones))

## Step 2. Intrinsic quality assessment, without a reference

Contributor counts, edit dates, and attribute richness, computed from OSM alone. When all three point the same way, suspect the completeness.

In [ ]:
intr = v.intrinsic_quality(bld, zones)
intr.head()

## Step 3. Extrinsic completeness, when a reference exists

There is a paradox here: if a reference existed, there would be less reason to depend on OSM. In practice the intrinsic indicators are usually what you have.

In [ ]:
truth = pd.read_csv("data/ground_truth.csv")
obs = bld.groupby("zone_id").size().rename("n_osm")
comp = truth.set_index("zone_id").join(obs).fillna(0)
comp["completeness"] = comp.n_osm / comp.buildings_true
print(comp.groupby("zone_type")["completeness"].mean().round(3).to_string())

## Step 4. Can intrinsic indicators predict completeness?

This decides what is possible in practice. Analysing without knowing completeness, and estimating it from a proxy while stating the limitation, are research of entirely different standards.

In [ ]:
contrib = bld.groupby("zone_id").n_contributors.mean().reindex(comp.index)
print("contributor count vs completeness, r =",
      round(float(np.corrcoef(contrib, comp.completeness)[0, 1]), 3))

## Step 5. Street network form

Orientation entropy separates planned from self-built fabric in one number. Street density needs more care: a low value may mean few streets, or that the lanes were never recorded.

In [ ]:
net = v.network_metrics(roads, zones)
merged = net.merge(truth[["zone_id", "zone_type"]], on="zone_id")
print(merged.groupby("zone_type")[["street_km_per_km2", "orientation_entropy"]]
      .mean().round(3).to_string())

## Step 6. Correction, and the central result of this chapter

Compare the density ranking before and after correction. Uncorrected, the densest area in the city can appear the sparsest.

In [ ]:
comp["adjusted"] = v.adjust_for_completeness(comp["n_osm"], comp["completeness"])
summary = comp.groupby("zone_type")[["n_osm", "adjusted", "buildings_true"]].mean().round(1)
summary.columns = ["OSM as-is", "corrected", "actual"]
print(summary.to_string())
print()
print("ranking from raw OSM:  ", list(summary["OSM as-is"].sort_values(ascending=False).index))
print("ranking after correction:", list(summary["corrected"].sort_values(ascending=False).index))

## Step 7. Export for mapping

Corrected figures are estimates and should be labelled as such: report the observed value, the estimated completeness, the corrected value, and the method used.

In [ ]:
out = zones.merge(comp[["completeness", "adjusted"]], left_on="zone_id",
                  right_index=True, how="left")
out.to_file("results.geojson", driver="GeoJSON")
print("saved -> results.geojson")

---

**What to do next.** Compare against Section 8.4. Before running this on your own city, check the OSM wiki for that country: **there is no standard tag for informal settlement.**